In [3]:
import cv2
from matplotlib import pyplot as plt

In [41]:
images = sorted(glob.glob('34759_final_project_raw/calib/image_03/data/*.png'))
templates = sorted(glob.glob('34759_final_project_raw/cropped_squares/*.png'))
assert images and templates

output_dir_stage1 = "stage1_removed"
os.makedirs(output_dir_stage1, exist_ok=True)

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    for tname in templates:
        template = cv2.imread(tname, cv2.IMREAD_GRAYSCALE)
        h, w = template.shape[:2]

        res = cv2.matchTemplate(gray, template, cv2.TM_CCORR_NORMED)
        _, _, _, max_loc = cv2.minMaxLoc(res)

        top_left = max_loc
        bottom_right = (top_left[0] + w, top_left[1] + h)

        cv2.rectangle(img, top_left, bottom_right, (255, 255, 255), -1)

    # Save the cleaned image
    base = os.path.basename(fname)
    cv2.imwrite(os.path.join(output_dir_stage1, base), img)


In [4]:
import os
import glob

def preprocess_for_corners(gray):

    # 1. Normalize lighting (excellent for masked regions)
    gray = cv2.equalizeHist(gray)

    # 2. Light Gaussian blur → helps corner detector
    gray = cv2.GaussianBlur(gray, (5,5), 0)

    # 3. Adaptive threshold → helps in uneven lighting
    thr = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY,
        11,  # block size
        3    # constant
    )

    # 4. Invert if needed (OpenCV prefers dark squares sometimes)
    # Count black/white ratio
    if np.mean(thr) > 127:
        thr = cv2.bitwise_not(thr)

    return thr
    

stage1_images = sorted(glob.glob(os.path.join('34759_final_project_raw/calib/image_02/data', '*.png')))
checkers = sorted(glob.glob('34759_final_project_raw/croppedCheckerboard/left/*.png'))
output_dir_sequence = "checker_sequence_left"
os.makedirs(output_dir_sequence, exist_ok=True)

# STEP 1 — Detect checker positions ONCE
img0 = cv2.imread(stage1_images[0])
gray0 = cv2.cvtColor(img0, cv2.COLOR_BGR2GRAY)

checker_positions = []

for cpath in checkers:
    temp = cv2.imread(cpath, cv2.IMREAD_GRAYSCALE)
    h, w = temp.shape[:2]
    res = cv2.matchTemplate(gray0, temp, cv2.TM_CCORR_NORMED)
    _, _, _, max_loc = cv2.minMaxLoc(res)
    top_left = max_loc
    bottom_right = (top_left[0] + w, top_left[1] + h)
    checker_positions.append((top_left, bottom_right))

# STEP 2 — Generate sequence
for i, fname in enumerate(stage1_images):
    img = cv2.imread(fname)

    # apply masking
    for idx, (top_left, bottom_right) in enumerate(checker_positions):
        if idx != i:
            cv2.rectangle(img, top_left, bottom_right, (255, 255, 255), -1)

    # convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # apply improved preprocessing
    proc = preprocess_for_corners(gray)

    # save processed image for calibration
    outname = os.path.join(output_dir_sequence, f"{i:05d}.png")
    cv2.imwrite(outname, proc)





In [1]:
import cv2 as cv
import glob
import numpy as np
 
def calibrate_camera(images_folder):
    images_names = sorted(glob.glob(images_folder))
    images = []
    for imname in images_names:
        im = cv.imread(imname, 1)
        images.append(im)
 
    #criteria used by checkerboard pattern detector.
    #Change this if the code can't find the checkerboard
    criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)
 
    rows = 5 #number of checkerboard rows.
    columns = 7 #number of checkerboard columns.
    world_scaling = 0.1 #change this to the real world square size. Or not.
 
    #coordinates of squares in the checkerboard world space
    objp = np.zeros((rows*columns,3), np.float32)
    objp[:,:2] = np.mgrid[0:columns, 0:rows].T.reshape(-1,2)
    objp = world_scaling* objp
 
    #frame dimensions. Frames should be the same size.
    width = images[0].shape[1]
    height = images[0].shape[0]
 
    #Pixel coordinates of checkerboards
    imgpoints = [] # 2d points in image plane.
 
    #coordinates of the checkerboard in checkerboard world space.
    objpoints = [] # 3d point in real world space
 
    for fname in images_names:
        img = cv.imread(fname)
        img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    
        ret, corners = cv.findChessboardCornersSB(img_gray, (columns, rows))
    
        if ret:
            # refine corners
            corners_sub = corners
    
            # Draw checkerboard corners on a copy
            img_draw = img.copy()
            cv.drawChessboardCorners(img_draw, (columns, rows), corners_sub, ret)
    
            # Show
            cv.imshow('Corners', img_draw)
            cv.waitKey(500)
    
            # Save for calibration
            objpoints.append(objp)
            imgpoints.append(corners_sub)

 
    ret, mtx, dist, rvecs, tvecs = cv.calibrateCamera(objpoints, imgpoints, (width, height), None, None)
    print(ret)
    return mtx, dist
 
mtx1, dist1 = calibrate_camera(images_folder = 'checker_sequence_left/*.png')
mtx2, dist2 = calibrate_camera(images_folder = 'checker_sequence_right/*.png')


[ERROR:1@0.282] global ocl.cpp:4677 createFromBinary OpenCL error CL_INVALID_VALUE (-30) during call: clCreateProgramWithBinary
[ERROR:2@0.283] global ocl.cpp:4677 createFromBinary OpenCL error CL_INVALID_VALUE (-30) during call: clCreateProgramWithBinary
[ERROR:0@0.285] global ocl.cpp:4677 createFromBinary OpenCL error CL_INVALID_VALUE (-30) during call: clCreateProgramWithBinary


0.2783440369480333
0.3361435934895202


In [2]:
import cv2 as cv
import glob
import numpy as np

class StereoCalibrator:
    def __init__(self, left_folder, right_folder, pattern=(7,5), square_size=0.1):
        self.left_images  = sorted(glob.glob(left_folder))
        self.right_images = sorted(glob.glob(right_folder))
        self.columns, self.rows = pattern  # (cols, rows)
        self.square_size = square_size

        # Prepare object points for each checkerboard
        self.objp = np.zeros((self.rows*self.columns,3), np.float32)
        self.objp[:,:2] = np.mgrid[0:self.columns, 0:self.rows].T.reshape(-1,2)
        self.objp *= square_size

        self.objpoints = []         # 3D points
        self.imgpoints_left = []    # 2D points in left image
        self.imgpoints_right = []   # 2D points in right image

    def find_corners(self):
        print("Detecting checkerboards...")
        for left_path, right_path in zip(self.left_images, self.right_images):
            imgL = cv.imread(left_path)
            imgR = cv.imread(right_path)

            grayL = cv.cvtColor(imgL, cv.COLOR_BGR2GRAY)
            grayR = cv.cvtColor(imgR, cv.COLOR_BGR2GRAY)

            retL, cornersL = cv.findChessboardCornersSB(grayL, (self.columns, self.rows))
            retR, cornersR = cv.findChessboardCornersSB(grayR, (self.columns, self.rows))

            if retL and retR:
                self.objpoints.append(self.objp)
                self.imgpoints_left.append(cornersL)
                self.imgpoints_right.append(cornersR)

        print(f"Found {len(self.objpoints)} valid stereo pairs.")

    def calibrate_intrinsics(self):
        # Use first left/right image for size
        img = cv.imread(self.left_images[0])
        h, w = img.shape[:2]

        print("Calibrating left camera...")
        retL, mtxL, distL, _, _ = cv.calibrateCamera(
            self.objpoints, self.imgpoints_left, (w,h), None, None)

        print("Calibrating right camera...")
        retR, mtxR, distR, _, _ = cv.calibrateCamera(
            self.objpoints, self.imgpoints_right, (w,h), None, None)

        print(f"Left RMS: {retL:.4f}, Right RMS: {retR:.4f}")
        self.mtxL, self.distL = mtxL, distL
        self.mtxR, self.distR = mtxR, distR

    def stereo_calibrate(self):
        img = cv.imread(self.left_images[0])
        h, w = img.shape[:2]

        flags = cv.CALIB_FIX_INTRINSIC
        ret, _, _, _, _, R, T, E, F = cv.stereoCalibrate(
            self.objpoints,
            self.imgpoints_left,
            self.imgpoints_right,
            self.mtxL, self.distL,
            self.mtxR, self.distR,
            (w,h),
            flags=flags
        )

        print(f"Stereo RMS error: {ret:.4f}")
        self.R, self.T, self.E, self.F = R, T, E, F

    def rectify(self):
        img = cv.imread(self.left_images[0])
        
        imgL = cv.imread("34759_final_project_raw/seq_01/image_02/data/0000000000.png")
        imgR = cv.imread("34759_final_project_raw/seq_01/image_03/data/0000000000.png")
        
        h, w = img.shape[:2]

        R1, R2, P1, P2, Q, roi1, roi2 = cv.stereoRectify(
            self.mtxL, self.distL,
            self.mtxR, self.distR,
            (w,h),
            self.R, self.T
        )

        self.map1x, self.map1y = cv.initUndistortRectifyMap(
            self.mtxL, self.distL, R1, P1, (w,h), cv.CV_32FC1)
        self.map2x, self.map2y = cv.initUndistortRectifyMap(
            self.mtxR, self.distR, R2, P2, (w,h), cv.CV_32FC1)
        self.Q = Q
        
        print("Stereo rectification complete.")

        rectL = cv.remap(imgL, self.map1x, self.map1y, cv.INTER_LINEAR)
        rectR = cv.remap(imgR, self.map2x, self.map2y, cv.INTER_LINEAR)

        # Draw epipolar lines
        line_color = (0,255,0)
        line_spacing = 25
        for y in range(0, h, line_spacing):
            cv.line(rectL, (0,y), (w,y), line_color, 1)
            cv.line(rectR, (0,y), (w,y), line_color, 1)

        cv.imshow("Rectified Left", rectL)
        cv.imshow("Rectified Right", rectR)
        cv.waitKey(0)
        cv.destroyAllWindows()

    def compute_disparity(self, idx=0):
        imgL = cv.imread(self.left_images[idx])
        imgR = cv.imread(self.right_images[idx])

        rectL = cv.remap(imgL, self.map1x, self.map1y, cv.INTER_LINEAR)
        rectR = cv.remap(imgR, self.map2x, self.map2y, cv.INTER_LINEAR)

        grayL = cv.cvtColor(rectL, cv.COLOR_BGR2GRAY)
        grayR = cv.cvtColor(rectR, cv.COLOR_BGR2GRAY)

        stereo = cv.StereoSGBM_create(
            minDisparity=0,
            numDisparities=64,
            blockSize=7,
            P1=8*3*7**2,
            P2=32*3*7**2,
            disp12MaxDiff=1,
            uniquenessRatio=10,
            speckleWindowSize=100,
            speckleRange=32
        )

        disp = stereo.compute(grayL, grayR).astype(np.float32)/16.0
        points_3D = cv.reprojectImageTo3D(disp, self.Q)

        # Show rectified images with epipolar lines
        cv.imshow("Rectified Left", rectL)
        cv.imshow("Rectified Right", rectR)
        cv.imshow("Disparity", (disp - disp.min()) / (disp.max() - disp.min()))
        cv.waitKey(0)
        cv.destroyAllWindows()

        return disp, points_3D

# --------------------------
# USAGE
# --------------------------
calib = StereoCalibrator(
    left_folder='checker_sequence_left/*.png',
    right_folder='checker_sequence_right/*.png',
    pattern=(7,5),
    square_size=0.1
)

calib.find_corners()
calib.calibrate_intrinsics()
calib.stereo_calibrate()
calib.rectify()

#disp, points_3D = calib.compute_disparity(idx=0)


Detecting checkerboards...


[ERROR:1@14.071] global ocl.cpp:4677 createFromBinary OpenCL error CL_INVALID_VALUE (-30) during call: clCreateProgramWithBinary
[ERROR:0@14.072] global ocl.cpp:4677 createFromBinary OpenCL error CL_INVALID_VALUE (-30) during call: clCreateProgramWithBinary
[ERROR:2@14.074] global ocl.cpp:4677 createFromBinary OpenCL error CL_INVALID_VALUE (-30) during call: clCreateProgramWithBinary


Found 7 valid stereo pairs.
Calibrating left camera...
Calibrating right camera...
Left RMS: 0.2783, Right RMS: 0.2715
Stereo RMS error: 1.6949
Stereo rectification complete.
